# meeting_extracted_w2_t3_label — W2 T3：Sales Brief 標註（規則，無 LLM）

- 用途：把中立層有效條目依「落在哪一期 Sales Brief 窗口」歸期，對到 Sales Brief 的 app / 品牌 / 四面向，判是否進對照（規格 §4.3、§7.1、§7.2）。一次標一期，整月重標覆寫；不呼叫 LLM（Phase 1 不做 L2 fallback）。
- 輸入 table：`{catalog}.{schema}.g_meeting_items_bu`（002）、`s_semantic_entity` / `s_semantic_entity_alias`（001；`app_default`、`sb_brand`）、`s_meeting_sb_facet_map`（004 種子）、`v_meeting_sb_period`（004；品牌清單與截止點）
- 輸出 table：`{catalog}.{schema}.g_meeting_items_sb_{unit}`（004）。MERGE on `(item_key, app_name)`；本期不在名單的舊列標 `active = false`。
- 參數（widgets）：`catalog`、`schema`、`unit`、`brief_month`（空 = 本月）、`window_mode`（`submitted_at` / `calendar` / `fixed_day`）、`fixed_day`、`grace_days`、`window_overrides`（JSON）、`labeler_version`（= [c02] `LABELER_RULE_VERSION_IN_CELL`）、`dry_run`
- 排程：每月一次（brief 填完後），接 T4。brief 還沒填時也能跑（截止點退回 `fixed_day`），填完再跑一次會用真實截止點整月重標。
- 負責人 / 更新日期：（填）/ 2026-10-23

Cell 標籤規則見 `docs/conventions.md`。

## 規則摘要（決定 D6 / D9 / D35）

| 步驟 | 規則 | 欄 |
|---|---|---|
| 窗口 | `submitted_at`：迄 = 該期 `v_meeting_sb_period.submitted_end`（缺 → 退回 `fixed_day` 並 WARN），起 = 上一期截止 + 1；兩端再加 `grace_days`。`calendar`：日曆月。`fixed_day`：每月 `fixed_day` 日。`window_overrides` 指定 `"APP:yyyy-MM": {"start","end"}` 最優先 | `brief_month` / `brief_month_basis` / `window_start` / `window_end` |
| app | 中立層 `app` 有值直接用；`ALL` / NULL → 客戶 `app_default` 展開多列；推不出 → `app_unresolved`（不進對照，只計數） | `app_name` / `app_basis_sb` |
| 品牌 | `customer` → 別名表 `sb_brand`；`samsung` 且 app ∈ {TV, MNT} 視同 `samsung_vd`；對不上保留 entity_id、`brand_in_sb = false` | `brand` / `brand_basis` / `brand_in_sb` |
| 面向 | `topic_tag_primary` 查映射（`rule_primary`）→ 主標籤不映射時看 `follow_tag`（schedule 隨 product_plan）與其餘 tag（`rule_secondary`）→ 都沒有 `unresolved` | `facet` / `facet_basis` |
| 相關 | 任一 tag 命中其 `relevance_rule`（always / spec_statement / capacity_timeref / quality_request_or_customer）即相關；再要求 app 已解、facet 非空 | `sb_relevant` / `relevance_basis` |


In [ ]:
# [c01] params
# W2 T3：一次標一期 brief_month。由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
import json as _json
import re as _re
from datetime import date as _date

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")                       # 讀寫哪一組：g_meeting_items_sb_{unit}
dbutils.widgets.text("brief_month", "")                  # yyyy-MM；空 = 本月
dbutils.widgets.dropdown("window_mode", "submitted_at", ["submitted_at", "calendar", "fixed_day"])  # D6
dbutils.widgets.text("fixed_day", "25")                  # fixed_day 模式的截止日；submitted_at 查不到截止點時也退回這個
dbutils.widgets.text("grace_days", "0")                  # 窗口兩端一起往後移幾天（brief 送出後幾天內的會議還算本期）
dbutils.widgets.text("window_overrides", "{}")           # JSON：{"TV:2026-08": {"start": "2026-07-26", "end": "2026-08-25"}}
dbutils.widgets.text("labeler_version", "0.1")           # 要等於 [c02] LABELER_RULE_VERSION_IN_CELL；改規則就升
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：算完印關卡與樣本，不寫表

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "brief_month": dbutils.widgets.get("brief_month").strip() or _date.today().strftime("%Y-%m"),
    "window_mode": dbutils.widgets.get("window_mode").strip(),
    "fixed_day": int(dbutils.widgets.get("fixed_day")),
    "grace_days": int(dbutils.widgets.get("grace_days")),
    "window_overrides": _json.loads(dbutils.widgets.get("window_overrides") or "{}"),
    "labeler_version": dbutils.widgets.get("labeler_version").strip(),
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert _re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", cfg["brief_month"]), "brief_month 格式 yyyy-MM"
assert 1 <= cfg["fixed_day"] <= 28 and 0 <= cfg["grace_days"] <= 15, "fixed_day 1–28；grace_days 0–15"
assert isinstance(cfg["window_overrides"], dict), "window_overrides 要是 JSON 物件"
print(cfg)


In [ ]:
# [c02] imports
# [c02]～[c04] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils），tests/ 直接載入。常數對齊 004 [c02] 與 001 種子。
import calendar
import json
from datetime import date, datetime, timedelta

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

LABELER_RULE_VERSION_IN_CELL = "0.1"   # 本 cell 規則版本；labeler_version 欄 = f"{rule}+map{map_version}"
APPS = ("TV", "NB", "MNT")
APP_UNRESOLVED = "app_unresolved"
WINDOW_MODES = ("submitted_at", "calendar", "fixed_day")
BASIS_OVERRIDE = "override"
FACETS = ("market_outlook", "product_tech", "price", "other")
RELEVANCE_RULES = ("always", "spec_statement", "capacity_timeref", "quality_request_or_customer", "never")
SPEC_STATEMENT_TYPES = ("plan", "decision", "request")   # §7.2 規則 2
# D9：Samsung 分不出事業部的 entity_id，在 TV / MNT 視同 Samsung VD（Sales Brief 的 Samsung）
SAMSUNG_GENERIC, SAMSUNG_VD, SAMSUNG_VD_APPS = "samsung", "samsung_vd", ("TV", "MNT")
SB_BRAND_ALIAS_TYPE = "sb_brand"

# 從中立層讀的欄（先過濾再 select；time_refs 只給規則 3 用，不寫進標註表）
ITEM_COLS = ["item_id", "item_key", "pk_seq", "segment_no", "sub_no", "unit", "customer", "app", "statement_type",
             "topic_tag_primary", "topic_tags", "speaker_side", "event_date", "summary", "sensitivity", "confidence",
             "time_refs"]
# 標註表欄（004 [c04] g_meeting_items_sb_bu 的順序；updated_at 由 [c04] to_label_rows 補）
LABEL_COLS = ["item_key", "app_name", "item_id", "pk_seq", "segment_no", "sub_no", "unit", "brief_month",
              "brief_month_basis", "window_start", "window_end", "app_basis_sb", "customer", "brand", "brand_basis",
              "brand_in_sb", "facet", "facet_basis", "sb_relevant", "relevance_basis", "statement_type",
              "topic_tag_primary", "topic_tags", "speaker_side", "event_date", "summary", "sensitivity", "confidence",
              "active", "labeler_version", "labeled_at", "updated_at"]


In [ ]:
# [c03] pure_window
# D6 窗口：純 Python 日期函式。periods[app][yyyy-MM] = 該期 submitted_end（datetime 或 None），來自 v_meeting_sb_period。
def month_add(ym: str, k: int) -> str:
    y, m = int(ym[:4]), int(ym[5:7])
    n = y * 12 + (m - 1) + k
    return f"{n // 12:04d}-{n % 12 + 1:02d}"


def month_bounds(ym: str) -> tuple[date, date]:
    y, m = int(ym[:4]), int(ym[5:7])
    return date(y, m, 1), date(y, m, calendar.monthrange(y, m)[1])


def fixed_cutoff(ym: str, day: int) -> date:
    """該月的截止日（day 超過月底就取月底）。"""
    y, m = int(ym[:4]), int(ym[5:7])
    return date(y, m, min(day, calendar.monthrange(y, m)[1]))


def _prev_submitted_end(app_periods: dict, brief_month: str) -> datetime | None:
    """brief_month 之前、最近一期有截止點的 submitted_end（期別可能跳月，例 TV 缺 2026-07）。"""
    before = [(ym, ts) for ym, ts in app_periods.items() if ym < brief_month and ts is not None]
    return max(before, key=lambda x: x[0])[1] if before else None


def app_window(app: str, brief_month: str, app_periods: dict, *, mode: str, fixed_day: int, grace_days: int,
               overrides: dict) -> dict:
    """→ {"start": date, "end": date, "basis": str, "note": str}。
    submitted_at：迄 = 本期 submitted_end 的日期；缺 → 退回 fixed_day（note 記 fallback）。起 = 上一期截止 + 1；
    上一期也缺 → 上個月 fixed_day + 1。兩端再加 grace_days。calendar：日曆月，不加 grace。fixed_day：每月 fixed_day。
    overrides["APP:yyyy-MM"] = {"start": "yyyy-MM-dd", "end": "yyyy-MM-dd"} 最優先。"""
    key = f"{app}:{brief_month}"
    if key in overrides:
        o = overrides[key]
        return {"start": date.fromisoformat(o["start"]), "end": date.fromisoformat(o["end"]),
                "basis": BASIS_OVERRIDE, "note": "override"}
    if mode == "calendar":
        s, e = month_bounds(brief_month)
        return {"start": s, "end": e, "basis": "calendar", "note": ""}
    g = timedelta(days=grace_days)
    prev_ts = _prev_submitted_end(app_periods, brief_month)
    start = (prev_ts.date() + timedelta(days=1)) if prev_ts is not None else (
        fixed_cutoff(month_add(brief_month, -1), fixed_day) + timedelta(days=1))
    note = "" if prev_ts is not None else "prev period missing → fixed_day start"
    if mode == "fixed_day":
        return {"start": start + g, "end": fixed_cutoff(brief_month, fixed_day) + g, "basis": "fixed_day", "note": note}
    end_ts = app_periods.get(brief_month)
    if end_ts is None:
        note = (note + "; " if note else "") + "no submitted_end → fixed_day fallback"
        return {"start": start + g, "end": fixed_cutoff(brief_month, fixed_day) + g, "basis": "fixed_day", "note": note}
    return {"start": start + g, "end": end_ts.date() + g, "basis": "submitted_at", "note": note}


def compute_windows(brief_month: str, periods: dict, *, mode: str, fixed_day: int, grace_days: int,
                    overrides: dict) -> dict[str, dict]:
    """每個 app（APPS）一個窗口；app_unresolved 用各 app 的最早起 ～ 最晚迄（不進對照，只為歸期計數）。"""
    assert mode in WINDOW_MODES, f"window_mode 只能是 {WINDOW_MODES}"
    out = {app: app_window(app, brief_month, periods.get(app, {}), mode=mode, fixed_day=fixed_day,
                           grace_days=grace_days, overrides=overrides) for app in APPS}
    for app, w in out.items():
        assert w["start"] <= w["end"], f"{app} 窗口起 > 迄：{w}"
    out[APP_UNRESOLVED] = {"start": min(w["start"] for w in out.values()), "end": max(w["end"] for w in out.values()),
                           "basis": "unresolved_span", "note": "各 app 窗口的聯集"}
    return out


def windows_to_rows(windows: dict[str, dict]) -> list[tuple]:
    """→ [(app_name, window_start, window_end, brief_month_basis)]，給 Spark 建小表 join。"""
    return [(app, w["start"], w["end"], w["basis"]) for app, w in windows.items()]


In [ ]:
# [c04] pure_label
# 純 DataFrame 函式：條目選取 → app 展開 → 窗口歸期 → 品牌 → brand_in_sb → 面向 → 相關性 → 標註表列。不 cache（serverless）。
def _empty_str_arr() -> F.Column:
    """空的 ARRAY<STRING> 常值（函式而不是模組常數：tests 載入 cell 時還沒有 SparkContext）。"""
    return F.expr("cast(array() AS ARRAY<STRING>)")


def _lit_map(d: dict) -> F.Column:
    """dict → map 常值（只放非 NULL 的值；查不到回 NULL）。空 dict → 空 map。"""
    items = [x for k, v in d.items() if v is not None for x in (F.lit(k), F.lit(v))]
    return F.create_map(*items) if items else F.expr("map()")


def select_items(df_items: DataFrame, unit: str, start: date, end: date) -> DataFrame:
    """有效條目（ok、未 superseded、本 unit、有 event_date）且 event_date 落在 [start, end]（所有 app 窗口的聯集；
    各 app 的精確窗口在 assign_window 再切）。一個 item_key 只留一列（item_id 最小），否則 MERGE 多源列。"""
    df = df_items.where(
        (F.col("unit") == F.lit(unit)) & (F.col("extract_status") == F.lit("ok")) & (F.col("superseded") == F.lit(False))
        & F.col("event_date").isNotNull() & F.col("event_date").between(F.lit(start), F.lit(end))
    ).select(*ITEM_COLS)
    w = Window.partitionBy("item_key").orderBy("item_id")
    return df.withColumn("_rn", F.row_number().over(w)).where(F.col("_rn") == 1).drop("_rn")


def expand_app(df: DataFrame, df_entity: DataFrame) -> DataFrame:
    """§4.3 第 2 點：app ∈ APPS 直接用（neutral）；ALL / NULL → 客戶 app_default 展開成多列（customer_default）；
    customer NULL 或 app_default 空 → 一列 app_unresolved（unresolved）。"""
    ent = df_entity.where(F.col("is_active")).select(F.col("entity_id").alias("customer"), F.col("app_default").alias("_apps"))
    j = df.join(ent, "customer", "left")
    has_app = F.col("app").isin(*APPS)
    defaults = F.coalesce(F.col("_apps"), _empty_str_arr())
    arr = (F.when(has_app, F.array(F.col("app")))
           .when(F.size(defaults) > 0, defaults)
           .otherwise(F.array(F.lit(APP_UNRESOLVED))))
    basis = (F.when(has_app, F.lit("neutral"))
             .when(F.size(defaults) > 0, F.lit("customer_default"))
             .otherwise(F.lit("unresolved")))
    return j.withColumn("app_basis_sb", basis).withColumn("app_name", F.explode(arr)).drop("_apps")


def assign_window(df: DataFrame, df_windows: DataFrame, brief_month: str) -> DataFrame:
    """df_windows: (app_name, window_start, window_end, brief_month_basis)。inner join 後只留 event_date 落在該 app 窗口內的列。"""
    j = df.join(df_windows, "app_name", "inner")
    return (j.where(F.col("event_date").between(F.col("window_start"), F.col("window_end")))
             .withColumn("brief_month", F.lit(brief_month)))


def assign_brand(df: DataFrame, df_alias: DataFrame) -> DataFrame:
    """customer → Sales Brief 品牌名（別名表 sb_brand）。D9：samsung 在 TV / MNT 視同 samsung_vd。
    對不上 → brand = customer（entity_id）、customer_fallback；customer NULL → none。"""
    ali = (df_alias.where((F.col("alias_type") == F.lit(SB_BRAND_ALIAS_TYPE)) & F.col("is_active"))
           .groupBy(F.col("entity_id").alias("_cust_eff")).agg(F.min("alias").alias("_sb_brand")))
    samsung = (F.col("customer") == F.lit(SAMSUNG_GENERIC)) & F.col("app_name").isin(*SAMSUNG_VD_APPS)
    d = df.withColumn("_cust_eff", F.when(samsung, F.lit(SAMSUNG_VD)).otherwise(F.col("customer")))
    j = d.join(ali, "_cust_eff", "left")
    basis = (F.when(F.col("customer").isNull(), F.lit("none"))
             .when(F.col("_sb_brand").isNull(), F.lit("customer_fallback"))
             .when(samsung, F.lit("samsung_vd_rule"))
             .otherwise(F.lit("sb_brand")))
    return (j.withColumn("brand", F.coalesce(F.col("_sb_brand"), F.col("customer")))
             .withColumn("brand_basis", basis).drop("_cust_eff", "_sb_brand"))


def assign_brand_in_sb(df: DataFrame, df_period: DataFrame, brief_month: str) -> DataFrame:
    """該期該 app 的 Sales Brief 品牌清單（v_meeting_sb_period.brands）有沒有這個 brand；沒有期別列或 brand NULL → false。"""
    per = (df_period.where(F.col("brief_month") == F.lit(brief_month))
           .select("app_name", F.col("brands").alias("_brands")))
    j = df.join(per, "app_name", "left")
    return (j.withColumn("brand_in_sb", F.coalesce(F.array_contains(F.col("_brands"), F.col("brand")), F.lit(False)))
             .drop("_brands"))


def assign_facet(df: DataFrame, facet_rows: list[dict]) -> DataFrame:
    """§7.1：主標籤直接命中 → rule_primary；主標籤不映射 → 先看它的 follow_tag 是否在 topic_tags，再看其餘 tag 依序第一個
    有映射的 → rule_secondary；都沒有 → facet NULL、unresolved。facet_rows = s_meeting_sb_facet_map 的 active 列。"""
    m_facet = _lit_map({r["topic_tag"]: r["facet"] for r in facet_rows})
    m_follow = _lit_map({r["topic_tag"]: r["follow_tag"] for r in facet_rows})
    tags = F.coalesce(F.col("topic_tags"), _empty_str_arr())
    primary = m_facet[F.col("topic_tag_primary")]
    follow = m_follow[F.col("topic_tag_primary")]
    follow_hit = F.when(follow.isNotNull() & F.array_contains(tags, follow), m_facet[follow])
    mapped = F.filter(F.transform(tags, lambda t: m_facet[t]), lambda x: x.isNotNull())
    secondary = F.when(F.size(mapped) > 0, F.element_at(mapped, 1))
    facet = F.coalesce(primary, follow_hit, secondary)
    basis = (F.when(primary.isNotNull(), F.lit("rule_primary"))
             .when(facet.isNotNull(), F.lit("rule_secondary"))
             .otherwise(F.lit("unresolved")))
    return df.withColumn("facet", facet).withColumn("facet_basis", basis)


def assign_relevance(df: DataFrame, facet_rows: list[dict]) -> DataFrame:
    """§7.2：任一 tag 依其 relevance_rule 命中即相關（relevance_basis = rule:tag，取 topic_tags 順序第一個）；
    沒命中 → 全部 tag 都是 never → never，否則 no_rule。sb_relevant 再要求 app 已解、facet 非空。"""
    m_rule = _lit_map({r["topic_tag"]: r["relevance_rule"] for r in facet_rows})
    tags = F.coalesce(F.col("topic_tags"), _empty_str_arr())
    time_refs_nonempty = (F.col("time_refs").isNotNull() & ~F.trim(F.col("time_refs")).isin("", "[]", "null"))

    def hit(t):
        rule = m_rule[t]
        return (F.when(rule == F.lit("always"), F.lit(True))
                .when(rule == F.lit("spec_statement"), F.col("statement_type").isin(*SPEC_STATEMENT_TYPES))
                .when(rule == F.lit("capacity_timeref"), time_refs_nonempty)
                .when(rule == F.lit("quality_request_or_customer"),
                      (F.col("statement_type") == F.lit("request")) | (F.col("speaker_side") == F.lit("customer")))
                .otherwise(F.lit(False)))

    hits = F.filter(tags, hit)
    first_hit = F.when(F.size(hits) > 0, F.element_at(hits, 1))
    n_ruled = F.size(F.filter(tags, lambda t: F.coalesce(m_rule[t], F.lit("never")) != F.lit("never")))
    basis = (F.when(first_hit.isNotNull(), F.concat(m_rule[first_hit], F.lit(":"), first_hit))
             .when(n_ruled == 0, F.lit("never"))
             .otherwise(F.lit("no_rule")))
    relevant = (first_hit.isNotNull() & (F.col("app_name") != F.lit(APP_UNRESOLVED)) & F.col("facet").isNotNull())
    return df.withColumn("relevance_basis", basis).withColumn("sb_relevant", relevant)


def to_label_rows(df: DataFrame, labeler_version: str) -> DataFrame:
    """→ 標註表欄（LABEL_COLS 順序）。active = true；labeled_at = updated_at = now。"""
    now = F.current_timestamp()
    d = (df.withColumn("active", F.lit(True)).withColumn("labeler_version", F.lit(labeler_version))
           .withColumn("labeled_at", now).withColumn("updated_at", now))
    return d.select(*LABEL_COLS)


def label_pipeline(df_items: DataFrame, df_entity: DataFrame, df_alias: DataFrame, df_period: DataFrame,
                   df_windows: DataFrame, facet_rows: list[dict], *, unit: str, brief_month: str,
                   span: tuple[date, date], labeler_version: str) -> DataFrame:
    """整條規則鏈；span = 所有窗口的 (最早起, 最晚迄)。"""
    df = select_items(df_items, unit, span[0], span[1])
    df = expand_app(df, df_entity)
    df = assign_window(df, df_windows, brief_month)
    df = assign_brand(df, df_alias)
    df = assign_brand_in_sb(df, df_period, brief_month)
    df = assign_facet(df, facet_rows)
    df = assign_relevance(df, facet_rows)
    return to_label_rows(df, labeler_version)


In [ ]:
# [c10] load_source
# 讀五個來源 → 算各 app 窗口（印出來看）→ 面向映射表 collect（14 列）。條目先用窗口聯集過濾再處理（大表先過濾）。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_items, fq_label = fq("g_meeting_items_bu"), fq(f"g_meeting_items_sb_{cfg['unit']}")
fq_entity, fq_alias = fq("s_semantic_entity"), fq("s_semantic_entity_alias")
fq_map, fq_period = fq("s_meeting_sb_facet_map"), fq("v_meeting_sb_period")
for t in (fq_items, fq_label, fq_entity, fq_alias, fq_map, fq_period):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 001 / 002 / 004"
assert cfg["labeler_version"] == LABELER_RULE_VERSION_IN_CELL, (
    f"widget labeler_version {cfg['labeler_version']} ≠ [c02] {LABELER_RULE_VERSION_IN_CELL}：兩邊一起改")

df_items = spark.table(fq_items)
df_entity, df_alias = spark.table(fq_entity), spark.table(fq_alias)
df_period = spark.table(fq_period)
facet_rows = [r.asDict() for r in spark.table(fq_map).where(F.col("is_active")).collect()]
assert facet_rows, f"{fq_map} 沒有 active 列：先跑 004"
map_version = (spark.table(fq("b_meeting_migration_log")).where(F.col("slug") == F.lit("004_meeting_sb_lens"))
               .orderBy(F.desc("run_at")).select("result").limit(1).collect())
map_version = (json.loads(map_version[0]["result"]).get("seed", {}).get("facet_map_version", "?") if map_version else "?")
labeler_version_full = f"{cfg['labeler_version']}+map{map_version}"

# 期別截止點：periods[app][yyyy-MM] = submitted_end
periods: dict[str, dict] = {}
for r in df_period.select("app_name", "brief_month", "submitted_end").collect():
    periods.setdefault(r["app_name"], {})[r["brief_month"]] = r["submitted_end"]
windows = compute_windows(cfg["brief_month"], periods, mode=cfg["window_mode"], fixed_day=cfg["fixed_day"],
                          grace_days=cfg["grace_days"], overrides=cfg["window_overrides"])
span = (windows[APP_UNRESOLVED]["start"], windows[APP_UNRESOLVED]["end"])
df_windows = spark.createDataFrame(windows_to_rows(windows),
                                   "app_name string, window_start date, window_end date, brief_month_basis string")
print(f"brief_month {cfg['brief_month']} | mode {cfg['window_mode']} | labeler {labeler_version_full}")
for app, w in windows.items():
    days = (w["end"] - w["start"]).days + 1
    flag = "  WARN " + w["note"] if w["note"] and app != APP_UNRESOLVED else ""
    print(f"  {app:<15} {w['start']} ~ {w['end']}  ({days:>3} 天) basis={w['basis']}{flag}")
has_brands = {r["app_name"] for r in df_period.where(F.col("brief_month") == F.lit(cfg["brief_month"]))
              .where(F.size("brands") > 0).select("app_name").collect()}
if set(APPS) - has_brands:
    print(f"WARN 本期 Sales Brief 還沒有這些 app 的品牌清單 {sorted(set(APPS) - has_brands)}：brand_in_sb 全 false，brief 填完後重跑")


In [ ]:
# [c11] run_label
# 跑整條規則鏈 → 本期標註列釘在 temp view（後面 MERGE / 驗證都用它，不重算）。
df_label = label_pipeline(df_items, df_entity, df_alias, df_period, df_windows, facet_rows, unit=cfg["unit"],
                          brief_month=cfg["brief_month"], span=span, labeler_version=labeler_version_full)
df_label.createOrReplaceTempView("_w2_t3_batch")
n_batch = df_label.count()
batch_keys = [(r["item_key"], r["app_name"]) for r in df_label.select("item_key", "app_name").collect()]
print(f"batch rows {n_batch}（item_key × app_name）")


In [ ]:
# [c12] gates
# 關卡只印不停（0 列也不停：brief 還沒填、或該期真的沒會議）。看：app_unresolved 比例（§4.3 第 2 點）、facet 推不出比例、
# brand 對不上 Sales Brief 比例、sb_relevant 比例、各 basis 分布，以及「格子預覽」（app × brand × facet 的相關條目數 = T4 會產的格）。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


def dist(col: str) -> dict:
    return {r[col]: r["count"] for r in df_label.groupBy(col).count().orderBy(F.desc("count")).collect()}


if n_batch == 0:
    print("本期 0 列：窗口內沒有有效條目（brief 還沒填？窗口對嗎？看 [c10] 印的窗口）。後面 cell 不寫表。")
else:
    def cnt(cond) -> F.Column:  # Spark 4 ANSI：boolean 不能直接 sum，先 cast int
        return F.sum(cond.cast("int"))

    agg = df_label.agg(
        cnt(F.col("app_name") == F.lit(APP_UNRESOLVED)).alias("unresolved"),
        cnt(F.col("facet").isNull()).alias("facet_null"),
        cnt(F.col("customer").isNull()).alias("customer_null"),
        cnt(~F.col("brand_in_sb") & F.col("customer").isNotNull()).alias("brand_not_in_sb"),
        cnt(F.col("sb_relevant")).alias("relevant"),
        F.countDistinct("item_key").alias("items"),
        F.countDistinct("pk_seq").alias("mails"),
    ).first()
    log = {
        "brief_month": cfg["brief_month"], "labeler_version": labeler_version_full, "rows": n_batch,
        "items": agg["items"], "mails": agg["mails"],
        "app_unresolved_rate": pct(agg["unresolved"], n_batch), "customer_null_rate": pct(agg["customer_null"], n_batch),
        "facet_null_rate": pct(agg["facet_null"], n_batch), "brand_not_in_sb_rate": pct(agg["brand_not_in_sb"], n_batch),
        "relevant": agg["relevant"], "relevant_rate": pct(agg["relevant"], n_batch),
        "app_dist": dist("app_name"), "app_basis_dist": dist("app_basis_sb"), "brand_basis_dist": dist("brand_basis"),
        "facet_dist": dist("facet"), "facet_basis_dist": dist("facet_basis"), "relevance_basis_dist": dist("relevance_basis"),
    }
    print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
    if (log["app_unresolved_rate"] or 0) > 0.3:
        print("WARN app_unresolved > 30%：中立層 app 空且客戶沒有 app_default（看 001 種子）或客戶 NULL")
    if (log["facet_null_rate"] or 0) > 0.3:
        print("WARN facet 推不出 > 30%：受控詞表或映射表（004 [c05]）要調（規格 §4.3 關卡）")
    print("\n格子預覽（sb_relevant 條目數；T4 的會議端格）：")
    (df_label.where(F.col("sb_relevant")).groupBy("app_name", "brand", "brand_in_sb", "facet").count()
     .orderBy("app_name", F.desc("count")).show(60, truncate=False))
    if cfg["dry_run"]:
        print("dry_run = true：不寫表。樣本（已 token 化，無人名）：")
        (df_label.orderBy(F.rand(7)).limit(12)
         .select("app_name", "app_basis_sb", "brand", "brand_basis", "brand_in_sb", "facet", "facet_basis", "sb_relevant",
                 "relevance_basis", "statement_type", "topic_tag_primary", "event_date", F.substring("summary", 1, 80).alias("summary_head"))
         .show(truncate=False))


In [ ]:
# [c20] write_label
# MERGE on (item_key, app_name)：本期的列全部 UPDATE / INSERT（整月重標覆寫，標註表沒有 PM 欄所以全欄都可蓋）。
# 同一個 key 若上次歸在別的期（窗口改了），這次直接改 brief_month，不會留兩列。
if not cfg["dry_run"] and n_batch > 0:
    upd = ", ".join(f"t.{c} = s.{c}" for c in LABEL_COLS if c not in ("item_key", "app_name", "labeled_at"))
    ins_cols, ins_vals = ", ".join(LABEL_COLS), ", ".join(f"s.{c}" for c in LABEL_COLS)
    spark.sql(f"""
        MERGE INTO {fq_label} t
        USING _w2_t3_batch s
        ON t.item_key = s.item_key AND t.app_name = s.app_name
        WHEN MATCHED THEN UPDATE SET {upd}, t.labeled_at = s.labeled_at
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)
    print(f"merged {n_batch} rows →", fq_label)
else:
    print("skip write（dry_run 或本期 0 列）")


In [ ]:
# [c21] maintain_label
# 整月重標的另一半：表裡 brief_month = 本期、active = true、但不在本批名單的列（條目被重萃取代、窗口改了掉出去、
# app 展開變了）→ active = false，不刪。本批 0 列時跳過（可能是 brief 還沒填、窗口暫時算錯，不能把整期停用）。
if not cfg["dry_run"] and n_batch > 0:
    n_deact = spark.sql(f"""
        SELECT count(*) AS n FROM {fq_label} t
        LEFT ANTI JOIN _w2_t3_batch s ON s.item_key = t.item_key AND s.app_name = t.app_name
        WHERE t.brief_month = '{cfg['brief_month']}' AND t.active = true
    """).first()["n"]
    if n_deact:
        spark.sql(f"""
            MERGE INTO {fq_label} t
            USING (SELECT t.item_key, t.app_name FROM {fq_label} t
                   LEFT ANTI JOIN _w2_t3_batch s ON s.item_key = t.item_key AND s.app_name = t.app_name
                   WHERE t.brief_month = '{cfg['brief_month']}' AND t.active = true) m
            ON t.item_key = m.item_key AND t.app_name = m.app_name
            WHEN MATCHED THEN UPDATE SET t.active = false, t.updated_at = current_timestamp()
        """)
    print(f"maintain: deactivated {n_deact}（本期不在名單的舊列）")
else:
    print("skip maintain（dry_run 或本期 0 列）")


In [ ]:
# [c30] check_target
# 寫入後驗證（用 [c11] 釘住的 batch_keys）：本批 key 都在表且版本對；(item_key, app_name) 無重複；本期 active 列數 = 本批；
# sb_relevant 的列 app 已解且 facet 非空；app_name / facet 值合法。最後印表的各期 active 列數與相關數。
if not cfg["dry_run"] and n_batch > 0:
    lab = spark.table(fq_label)
    keys_df = spark.createDataFrame(batch_keys, "item_key string, app_name string")
    got = lab.join(keys_df, ["item_key", "app_name"], "left_semi")
    this_month = lab.where((F.col("brief_month") == F.lit(cfg["brief_month"])) & F.col("active"))
    chk = {
        "batch_rows_in_table": got.count(), "expected": n_batch,
        "batch_version_ok": got.where(F.col("labeler_version") == F.lit(labeler_version_full)).count(),
        "dup_key": lab.groupBy("item_key", "app_name").count().where("count > 1").count(),
        "month_active": this_month.count(),
        "relevant_bad": got.where(F.col("sb_relevant") & (F.col("facet").isNull() | (F.col("app_name") == F.lit(APP_UNRESOLVED)))).count(),
        "bad_app": got.where(~F.col("app_name").isin(*APPS, APP_UNRESOLVED)).count(),
        "bad_facet": got.where(F.col("facet").isNotNull() & ~F.col("facet").isin(*FACETS)).count(),
        "table_total": lab.count(),
        "active_by_month": {r["brief_month"]: (r["n"], r["relevant"]) for r in
                            lab.where(F.col("active")).groupBy("brief_month")
                            .agg(F.count("*").alias("n"), F.sum(F.col("sb_relevant").cast("int")).alias("relevant"))
                            .orderBy("brief_month").collect()},
    }
    print(chk)
    assert chk["batch_rows_in_table"] == chk["expected"] == chk["batch_version_ok"], "本批 key 沒全寫進表或版本不對"
    assert chk["dup_key"] == 0, "(item_key, app_name) 重複（MERGE 鍵壞了）"
    assert chk["month_active"] == n_batch, "本期 active 列數 ≠ 本批（[c21] 停用沒做完）"
    assert chk["relevant_bad"] == 0 and chk["bad_app"] == 0 and chk["bad_facet"] == 0, "sb_relevant / app_name / facet 不合法"
